In [2]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# RAG tool 
from langchain_community.document_loaders import JSONLoader
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.runnables import RunnablePassthrough

In [ ]:
# load menuu 
loader = JSONLoader(file_path= 'data/menu.json', jq_schema= ".[]", text_content= False)
docs = loader.load() 

# Split the chunks 
splitter = RecursiveCharacterTextSplitter(chunk_size= 500, chunk_overlap=50)
chunks = splitter.split_documents(docs) 

# Embed and store 
embeddings = OllamaEmbeddings(model= 'llama3')
vectorstore = Chroma.from_documents(chunks, embeddings, persist_directory= "./db/menu_chroma")

# Build RAG 
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
rag_prompt = ChatPromptTemplate.from_template("""
Bạn là nhân viên phục vụ nhà hàng. Dựa vào thực đơn dưới đây, hãy trả lời câu hỏi của khách.
Hãy trả lời bằng tiếng việt

Thực đơn:
{context}

Câu hỏi: {question}
""")


def format_docs(docs):
    return "\\n\\n".join(d.page_content for d in docs)

llm = ChatOllama(model= 'llama3.1')
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

answer = rag_chain.invoke("Bạn có món bò lúc lắc không?")
print(answer)

Xin chào khách! Chúng tôi không có món bò lúc lắc trên thực đơn của chúng tôi. Món Cà Phê Muối Huệ là một loại trà được pha trộn với tinh chất Robusta, kem súa và mè, rất phù hợp cho những người yêu thích hương vị ngọt ngào. Nếu bạn đang tìm kiếm món ăn khác, vui lòng hỏi thêm!


In [6]:
from langchain_core.tools import tool

@tool
def create_order(table_number: int, item_name: str, quantity: int) -> str:
    """Creates a new food order for a table. Use when customer wants to order food."""
    # Call your actual Order API here
    return f"Đã đặt {quantity} phần {item_name} cho bàn {table_number}."

@tool
def get_bill(table_number: int) -> str:
    """Gets the total bill for a table. Use when customer asks for the check."""
    # Call your Payment API here
    return f"Tổng tiền bàn {table_number}: 250,000 VND."

# Bind tools to the LLM
llm_with_tools = llm.bind_tools([create_order, get_bill])

In [10]:
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

llm = ChatOllama(model='llama3.1')
tools = [create_order, get_bill]

agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "Bạn là nhân viên phục vụ nhà hàng AI. Gọi tool khi cần xử lý đơn hàng."),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])

agent = create_tool_calling_agent(llm, tools, agent_prompt)

# AgentExecutor handles the actual loop: LLM -> Tool -> LLM -> Output
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=False,           # See the inner "thoughts" of the agent
    max_iterations=5,       # Stop runaway loops
    handle_parsing_errors=True
)

# Example 1: Single Tool
result = agent_executor.invoke({"input": "Cho bàn 3 một tô phở bò"})
print(result["output"])

# Example 2: Parallel Tool Calling
# The LLM will call create_order for table 5 AND getting the bill for table 2 at the same time
result = agent_executor.invoke({
    "input": "Cậu kêu 1 tô phở cho bàn 5, à quên tính tiền bàn 2 giúp mình với"
})
print(result["output"])

{"status_code":200,"message":"Order created successfully"}
Đã tính toán được tổng tiền bàn 2 là 250,000 VND.


In [12]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

# 1. Define the model
llm = ChatOllama(model="llama3.1")

# 2. Define a prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly Vietnamese restaurant waiter. Answer in Vietnamese."),
    ("human", "{input}")
])

# 3. Build a chain (prompt → model)
chain = prompt | llm

# 4. Invoke
response = chain.invoke({"input": "Cho tôi xem thực đơn"})
print(response.content)

Dưới đây là thực đơn của chúng tôi:

**Món Ngon**

* Bún Tháng: Món phổ biến của Hà Nội, gồm bún tươi, thịt heo nướng, trứng ốp la và nước mắm pha.
* Phở Bò: Sợi phở mềm, nước dùng đậm vị, thịt bò mềm ngon.
* Gỏi Cuốn: Rau sống cuốn với bánh tráng, thịt heo, tôm, và các loại gia vị.

**Món Mặn**

* Cơm Tấm: Cơm vàng thơm, được trang trí với trứng ốp la, thịt heo nướng, và nước mắm pha.
* Bánh Xèo: Bánh giòn rụm, chứa đầy thịt heo, tôm, và rau sống.
* Gà Rán: Gà tươi được ướp gia vị và chiên giòn.

**Món Hải Sản**

* Cá Chép Nướng: Cá chép nướng với các loại gia vị đặc trưng của Việt Nam.
* Tôm Hùm Chiên: Tôm hùm tươi, được ướp gia vị và chiên giòn.
* Mực Xào: Mực tươi xào với tỏi, ớt và rau sống.

**Trái Cây và Trà**

* Nước Mắm Pha: Nước mắm pha đặc trưng của Việt Nam.
* Rau Sắt Lá (Cải Xanh): Rau cải xanh được muối chua ngọt.
* Trái Cây (Đu Đủ, Táo, Bưởi)

Bạn muốn thử món gì?


In [20]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Define the RAG logic
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 2. Create the prompt with BOTH Context (Menu) and History (Memory)
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Bạn là người phục vụ tại nhà hàng. Chỉ sử dụng Menu dưới đây để trả lời.\n\nMENU:\n{context}"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

# 3. Build the full chain
rag_chain = (
    {"context": vectorstore.as_retriever() | format_docs, "input": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser() # This automatically extracts the .content for you!
)

# 4. Wrap with Memory
final_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)


In [22]:
from langchain_community.document_loaders import JSONLoader
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
import os

# --- 1. SETUP VECTOR STORE (RETRIEVAL) ---
loader = JSONLoader(file_path="data/menu.json", jq_schema=".[]", text_content=False)
docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)
embeddings = OllamaEmbeddings(model="llama3.1") # Use llama3.1 for tool/structured support
vectorstore = Chroma.from_documents(chunks, embeddings, persist_directory="./db/menu_chroma")
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# --- 2. SETUP MEMORY STORE ---
store = {}

def get_session_history(session_id: str) -> ChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

from operator import itemgetter
# --- 3. SETUP THE RAG CHAIN (Fixed) ---
llm = ChatOllama(model="llama3.1")
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Bạn là người phục vụ nhà hàng thông minh. Hãy sử dụng thông tin MENU dưới đây để trả lời khách hàng.\n\nMENU:\n{context}"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)
# We use itemgetter to extract specific keys from the memory dictionary
rag_chain = (
    {
        "context": itemgetter("input") | retriever | format_docs, # Extracts only text for search
        "input": itemgetter("input"),                             # Extracts text for prompt
        "history": itemgetter("history"),                         # Extracts history for prompt
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)
# --- 4. WRAP WITH HISTORY ---
final_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

# --- 5. INTERACT ---
# First Turn
print("--- LẦN 1 ---")
response1 = final_chain.invoke(
    {"input": "Cho tôi món nào dòn dòn và có tôm?"},
    config={"configurable": {"session_id": "Table_10"}}
)
print(f"AI: {response1}\n")

# Second Turn (Memory Test)
print("--- LẦN 2 ---")
response2 = final_chain.invoke(
    {"input": "Món đó giá bao nhiêu?"},
    config={"configurable": {"session_id": "Table_10"}}
)
print(f"AI: {response2}")


--- LẦN 1 ---
AI: Khách ơi! Bạn muốn thử món gì dòn dòn và có tôm không?

Tôi nghĩ món Tôm sẽ phù hợp với yêu cầu của bạn. Tôm tươi được nướng lên, có vị ngọt và dai dai rất ngon!

Hãy cho tôi biết nếu bạn muốn đặt món Tôm này nhé!

--- LẦN 2 ---
AI: Khách ơi! Món Tôm của chúng tôi có giá khoảng 120.000 VND.

Tuy nhiên, giá có thể thay đổi tùy thuộc vào mùa và nguồn gốc của tôm. Nhưng với món Tôm hiện tại, giá sẽ là 120.000 VND/phần.
